# Lanzarote + Fuerteventura — Calima Analysis
**USO INTERNO** | `notebooks/lanzaftv_calima_analysis.ipynb`

Análisis calima-mortalidad para la unidad insular Lanzarote+Fuerteventura,
aplicando Proxy v5 calibrado en Gran Canaria (AUC=0.917).

⚠️ **Nota metodológica:** Proxy v5 fue calibrado con datos de GC.
Su aplicación aquí asume transferibilidad del modelo entre islas —
tratar resultados como exploratorios hasta validación con ground truth propio.

| Item | Value |
|---|---|
| Unit | Lanzarote + Fuerteventura (combined) |
| Master | `data/processed/lanzaftv/master/master_lztftv_2004_2025.parquet` |
| Period | 2009–2025 (regression) |
| Proxy | v5 transferred from Gran Canaria (AUC=0.917) |
| Model | OLS HC3 — same spec as GC deep dive |

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

BASE = r"C:\Users\fdora\RA_Career\Projects\climate_mortality"

# ── RECALIBRAR PROXY V5 (calibrado en GC) ─────────────────────────────
df_gc = pd.read_parquet(f"{BASE}/data/processed/gran_canaria/master/master_gcan_2004_2025.parquet")
df_gc['week_start'] = pd.to_datetime(df_gc['week_start'])

cal = df_gc[(df_gc['week_start'] >= '2018-06-18') & (df_gc['week_start'] <= '2022-03-14')].copy()
cal['gt'] = ((cal['calima_dai_flag'] == 1) | (cal['cap_dust_yellow_plus_week'] == 1)).astype(int)

features = ['PM10', 'PM2.5', 'vis_min_m_week']
scaler = MinMaxScaler()
X_cal = scaler.fit_transform(cal[features].values)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_cal, cal['gt'].values)
print(f"Proxy v5 recalibrado — AUC: {roc_auc_score(cal['gt'].values, lr.predict_proba(X_cal)[:,1]):.3f}")

# ── LOAD LANZAROTE+FUERTEVENTURA ───────────────────────────────────────
df_lzt = pd.read_parquet(f"{BASE}/data/processed/lanzaftv/master/master_lztftv_2004_2025.parquet")
df_lzt['week_start'] = pd.to_datetime(df_lzt['week_start'])
df_lzt = df_lzt.sort_values('week_start').reset_index(drop=True)

df_lzt['vis_min_m_week_imp'] = df_lzt['vis_min_m_week'].fillna(df_lzt['vis_min_m_week'].median())
df_lzt['deaths_week'] = df_lzt['deaths_week'].fillna(df_lzt['deaths_week'].median())

# Aplicar proxy v5
X_lzt = scaler.transform(df_lzt[['PM10', 'PM2.5', 'vis_min_m_week_imp']].values)
df_lzt['calima_v5_score'] = lr.predict_proba(X_lzt)[:, 1]
df_lzt['calima_v5_q'] = pd.cut(
    df_lzt['calima_v5_score'],
    bins=[-0.001, 0.104, 0.227, 0.422, 1.001],
    labels=[0, 1, 2, 3]
).astype(float)

# Filtrar 2009-2025 + variables
df_reg_lzt = df_lzt[df_lzt['week_start'] >= '2009-01-01'].copy().reset_index(drop=True)
df_reg_lzt['deaths_lag1']      = df_reg_lzt['deaths_week'].shift(1)
df_reg_lzt['calima_v5_q_lag1'] = df_reg_lzt['calima_v5_q'].shift(1)
df_reg_lzt['calima_v5_q_lag2'] = df_reg_lzt['calima_v5_q'].shift(2)
df_reg_lzt.dropna(subset=['deaths_lag1', 'calima_v5_q_lag2'], inplace=True)

print(f"Dataset: {df_reg_lzt.shape[0]} weeks")
print(f"\nCalima v5 distribution:")
print(df_reg_lzt['calima_v5_q'].value_counts().sort_index())
print(f"\nNulls: {df_reg_lzt[['deaths_week','calima_v5_q','temp_c_mean','deaths_lag1']].isnull().sum().to_dict()}")

Proxy v5 recalibrado — AUC: 0.917
Dataset: 885 weeks

Calima v5 distribution:
calima_v5_q
0.0    167
1.0    156
2.0    224
3.0    338
Name: count, dtype: int64

Nulls: {'deaths_week': 0, 'calima_v5_q': 0, 'temp_c_mean': 2, 'deaths_lag1': 0}


In [5]:
print("CAP dust:")
cap = df_lzt[df_lzt['cap_dust_yellow_plus_week'].notna()]
print(f"  Desde: {cap['week_start'].min().date()}")
print(f"  Hasta: {cap['week_start'].max().date()}")
print(f"  Semanas: {len(cap)}")

print("\nDAI flag:")
dai = df_lzt[df_lzt['calima_dai_flag'].notna()]
print(f"  Desde: {dai['week_start'].min().date()}")
print(f"  Hasta: {dai['week_start'].max().date()}")
print(f"  Semanas: {len(dai)}")

print("\nOverlap:")
both = df_lzt[df_lzt['calima_dai_flag'].notna() & df_lzt['cap_dust_yellow_plus_week'].notna()]
print(f"  Desde: {both['week_start'].min().date()}")
print(f"  Hasta: {both['week_start'].max().date()}")
print(f"  Semanas: {len(both)}")

# Distribución de eventos positivos en overlap
cal_lzt = both.copy()
cal_lzt['gt'] = ((cal_lzt['calima_dai_flag'] == 1) | (cal_lzt['cap_dust_yellow_plus_week'] == 1)).astype(int)
print(f"\nGround truth en overlap:")
print(cal_lzt['gt'].value_counts())
print(f"Positive rate: {cal_lzt['gt'].mean():.1%}")

CAP dust:
  Desde: 2018-06-18
  Hasta: 2025-12-29
  Semanas: 394

DAI flag:
  Desde: 2003-12-29
  Hasta: 2022-03-14
  Semanas: 951

Overlap:
  Desde: 2018-06-18
  Hasta: 2022-03-14
  Semanas: 196

Ground truth en overlap:
gt
0    186
1     10
Name: count, dtype: int64
Positive rate: 5.1%


In [7]:
features_lzt = ['PM10', 'vis_min_m_week']

X_cal_lzt = cal_lzt[features_lzt].values
y_cal_lzt = cal_lzt['gt'].values

scaler_lzt = MinMaxScaler()
X_cal_lzt_scaled = scaler_lzt.fit_transform(X_cal_lzt)

lr_lzt = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_lzt.fit(X_cal_lzt_scaled, y_cal_lzt)

probs_lzt = lr_lzt.predict_proba(X_cal_lzt_scaled)[:, 1]
auc_lzt = roc_auc_score(y_cal_lzt, probs_lzt)

print(f"AUC (calibration 2018-06 → 2022-03): {auc_lzt:.3f}")
print(f"n_positives: {y_cal_lzt.sum()} | n_total: {len(y_cal_lzt)} | EPV: {y_cal_lzt.sum()/2:.1f}")
print(f"\nCoefficients:")
for feat, coef in zip(features_lzt, lr_lzt.coef_[0]):
    print(f"  {feat:20s}: {coef:+.4f}")
print(f"  {'Intercept':20s}: {lr_lzt.intercept_[0]:+.4f}")

# Comparación con GC
print(f"\n--- vs Proxy v5 GC (3 features) ---")
print(f"  GC AUC:      0.917 | positives: 17 | EPV: 5.7")
print(f"  LZT+FTV AUC: {auc_lzt:.3f} | positives: 10 | EPV: {y_cal_lzt.sum()/2:.1f}")
print(f"\n⚠️  EPV=5 — resultados exploratorios, interpretar con cautela")

AUC (calibration 2018-06 → 2022-03): 0.896
n_positives: 10 | n_total: 196 | EPV: 5.0

Coefficients:
  PM10                : +3.7028
  vis_min_m_week      : -2.4062
  Intercept           : +0.1345

--- vs Proxy v5 GC (3 features) ---
  GC AUC:      0.917 | positives: 17 | EPV: 5.7
  LZT+FTV AUC: 0.896 | positives: 10 | EPV: 5.0

⚠️  EPV=5 — resultados exploratorios, interpretar con cautela


In [8]:
# Aplicar proxy LZT+FTV al dataset completo
df_lzt['vis_min_m_week_imp'] = df_lzt['vis_min_m_week'].fillna(df_lzt['vis_min_m_week'].median())

X_full_lzt = scaler_lzt.transform(df_lzt[['PM10', 'vis_min_m_week_imp']].values)
df_lzt['calima_v5_score'] = lr_lzt.predict_proba(X_full_lzt)[:, 1]

# Cuartiles propios de LZT+FTV
df_reg_lzt = df_lzt[df_lzt['week_start'] >= '2009-01-01'].copy().reset_index(drop=True)
q_lzt = df_reg_lzt['calima_v5_score'].quantile([0.25, 0.50, 0.75])
print(f"Cuartiles score LZT+FTV: {q_lzt.round(3).to_dict()}")

df_reg_lzt['calima_v5_q'] = pd.cut(
    df_reg_lzt['calima_v5_score'],
    bins=[-0.001, q_lzt[0.25], q_lzt[0.50], q_lzt[0.75], 1.001],
    labels=[0, 1, 2, 3]
).astype(float)

# Variables regresión
df_reg_lzt['temp_c_mean'] = df_reg_lzt['temp_c_mean'].fillna(df_reg_lzt['temp_c_mean'].median())
df_reg_lzt['deaths_lag1']      = df_reg_lzt['deaths_week'].shift(1)
df_reg_lzt['calima_v5_q_lag1'] = df_reg_lzt['calima_v5_q'].shift(1)
df_reg_lzt['calima_v5_q_lag2'] = df_reg_lzt['calima_v5_q'].shift(2)
df_reg_lzt.dropna(subset=['deaths_lag1', 'calima_v5_q_lag2'], inplace=True)

print(f"\nDataset: {df_reg_lzt.shape[0]} weeks")
print(f"Calima distribution: {df_reg_lzt['calima_v5_q'].value_counts().sort_index().to_dict()}")

# Modelos lag0, lag1, lag2
print("\n=== REGRESIÓN — Lanzarote+Fuerteventura (Proxy v5 local) ===\n")
for label, var in [
    ("Lag 0 (contemporáneo)", 'calima_v5_q'),
    ("Lag 1 (1 semana)",      'calima_v5_q_lag1'),
    ("Lag 2 (2 semanas)",     'calima_v5_q_lag2'),
]:
    m = smf.ols(f"deaths_week ~ {var} + temp_c_mean + deaths_lag1", data=df_reg_lzt).fit(cov_type='HC3')
    b = m.params[var]; p = m.pvalues[var]
    ci = m.conf_int().loc[var]
    dw = durbin_watson(m.resid)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '—'
    print(f"{label:25s}  β={b:+.3f}  p={p:.4f} {sig}  CI=[{ci[0]:.2f},{ci[1]:.2f}]  R²={m.rsquared:.3f}  DW={dw:.2f}")

Cuartiles score LZT+FTV: {0.25: 0.223, 0.5: 0.317, 0.75: 0.38}

Dataset: 885 weeks
Calima distribution: {0.0: 222, 1.0: 222, 2.0: 221, 3.0: 220}

=== REGRESIÓN — Lanzarote+Fuerteventura (Proxy v5 local) ===

Lag 0 (contemporáneo)      β=+0.091  p=0.6018 —  CI=[-0.25,0.43]  R²=0.283  DW=2.33
Lag 1 (1 semana)           β=-0.069  p=0.6901 —  CI=[-0.41,0.27]  R²=0.283  DW=2.33
Lag 2 (2 semanas)          β=+0.233  p=0.1856 —  CI=[-0.11,0.58]  R²=0.284  DW=2.33
